In [1]:
# ============================================================
# WEEK 1 — MDP Design & Custom Gym Environment
# Project 2: Travel & Hospitality — RL Dynamic Pricing
# Intern Branch: preeti-dev | Infotact Solutions
#
# UNIQUE FEATURES ADDED:
#   ★ External Market Events (holiday surge, competitor sale, weather)
#   ★ Three Customer Segments (business, leisure, last-minute)
#   ★ Competitor Pricing Signal in State Space
#   ★ Dual Seat Class (Economy + Business)
# ============================================================
 
 
# ── CELL 1: Install & Import Libraries ───────────────────
import subprocess, sys
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
 
for lib in ["gymnasium", "numpy", "matplotlib", "seaborn", "pandas"]:
    install(lib)
 
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')
 
os.makedirs('../reports', exist_ok=True)
os.makedirs('../models',  exist_ok=True)
os.makedirs('../data',    exist_ok=True)
 
print("✅ All libraries imported successfully")
print(f"   Gymnasium version : {gym.__version__}")
 

✅ All libraries imported successfully
   Gymnasium version : 1.3.0


In [2]:
# ── CELL 2: MDP Problem Formulation ──────────────────────
print("=" * 60)
print("   ENHANCED MDP — Contextual Airline Dynamic Pricing")
print("=" * 60)
print()
print("  SCENARIO:")
print("  ─────────")
print("  An airline manages Economy + Business seats on a flight")
print("  departing in 30 days. Each day it sets prices for both")
print("  classes, facing 3 customer segments and random market")
print("  events that shift demand unexpectedly.")
print()
print("  STANDARD MDP COMPONENTS:")
print("  ─────────────────────────")
print("  State  : [eco_seats, biz_seats, days_left,")
print("            competitor_price_idx, market_event_id]")
print("  Action : [economy_price_idx, business_price_idx]")
print("  Reward : total daily revenue (eco + biz combined)")
print("  Done   : days_left = 0  OR  both classes sold out")
print()
print("  ★ UNIQUE ADDITIONS vs standard RL pricing projects:")
print("  ─────────────────────────────────────────────────────")
print("  ★ Market Events  : holiday/competitor sale/bad weather")
print("  ★ 3 Segments     : business / leisure / last-minute")
print("  ★ Competitor     : rival airline price in state space")
print("  ★ Dual Class     : Economy + Business managed together")
print()
print("✅ MDP formulation complete")


   ENHANCED MDP — Contextual Airline Dynamic Pricing

  SCENARIO:
  ─────────
  An airline manages Economy + Business seats on a flight
  departing in 30 days. Each day it sets prices for both
  classes, facing 3 customer segments and random market
  events that shift demand unexpectedly.

  STANDARD MDP COMPONENTS:
  ─────────────────────────
  State  : [eco_seats, biz_seats, days_left,
            competitor_price_idx, market_event_id]
  Action : [economy_price_idx, business_price_idx]
  Reward : total daily revenue (eco + biz combined)
  Done   : days_left = 0  OR  both classes sold out

  ★ UNIQUE ADDITIONS vs standard RL pricing projects:
  ─────────────────────────────────────────────────────
  ★ Market Events  : holiday/competitor sale/bad weather
  ★ 3 Segments     : business / leisure / last-minute
  ★ Competitor     : rival airline price in state space
  ★ Dual Class     : Economy + Business managed together

✅ MDP formulation complete


In [3]:
# ── CELL 3: Market Event System ──────────────────────────
# Each day, one of 4 possible market conditions applies.
# This is the first unique feature — real airline pricing
# reacts to external shocks, not just internal inventory.
 
MARKET_EVENTS = {
    0: {'name': 'Normal',          'demand_multiplier': 1.0,  'color': '#60a5fa'},
    1: {'name': 'Holiday Surge',   'demand_multiplier': 1.6,  'color': '#34d399'},
    2: {'name': 'Competitor Sale', 'demand_multiplier': 0.6,  'color': '#f87171'},
    3: {'name': 'Bad Weather',     'demand_multiplier': 0.75, 'color': '#fbbf24'},
}
 
# Probability of each event occurring on any given day
EVENT_PROBS = [0.65, 0.12, 0.15, 0.08]  # must sum to 1.0
 
print("✅ Market Event System defined")
print()
print(f"  {'Event':<22} {'Demand Multiplier':>18} {'Probability':>12}")
print("  " + "-" * 54)
for idx, ev in MARKET_EVENTS.items():
    print(f"  {ev['name']:<22} {ev['demand_multiplier']:>18.1f}×"
          f" {EVENT_PROBS[idx]:>11.0%}")


✅ Market Event System defined

  Event                   Demand Multiplier  Probability
  ------------------------------------------------------
  Normal                                1.0×         65%
  Holiday Surge                         1.6×         12%
  Competitor Sale                       0.6×         15%
  Bad Weather                           0.8×          8%


In [4]:
# ── CELL 4: Customer Segment System ──────────────────────
# Three distinct customer types exist simultaneously.
# Each has different price sensitivity and booking timing.
# This is the second unique feature.
 
CUSTOMER_SEGMENTS = {
    'business': {
        'price_sensitivity' : 0.3,   # low — will pay high prices
        'booking_window'    : 'any', # books anytime
        'base_probability'  : 0.25,  # 25% of daily traffic
        'color'             : '#a78bfa'
    },
    'leisure': {
        'price_sensitivity' : 0.9,   # high — very price conscious
        'booking_window'    : 'early',# books far in advance
        'base_probability'  : 0.50,  # 50% of daily traffic
        'color'             : '#60a5fa'
    },
    'last_minute': {
        'price_sensitivity' : 0.2,   # very low — desperate, pays anything
        'booking_window'    : 'late', # only books in last 7 days
        'base_probability'  : 0.25,  # 25% of daily traffic
        'color'             : '#f87171'
    }
}
 
print("✅ Customer Segment System defined")
print()
print(f"  {'Segment':<15} {'Price Sensitivity':>18} "
      f"{'Base Prob':>10} {'When They Book':>15}")
print("  " + "-" * 60)
for name, seg in CUSTOMER_SEGMENTS.items():
    print(f"  {name:<15} {seg['price_sensitivity']:>18.1f} "
          f"{seg['base_probability']:>10.0%} {seg['booking_window']:>15}")


✅ Customer Segment System defined

  Segment          Price Sensitivity  Base Prob  When They Book
  ------------------------------------------------------------
  business                       0.3        25%             any
  leisure                        0.9        50%           early
  last_minute                    0.2        25%            late


In [ ]:
# ── CELL 5: Full Enhanced Gym Environment ────────────────
class ContextualAirlinePricingEnv(gym.Env):
    """
    Enhanced Custom Gymnasium Environment for Airline Seat Pricing.
 
    UNIQUE vs standard RL pricing environments:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ★ Dual class inventory (Economy + Business)
    ★ 3 customer segments with different behaviours
    ★ Random market events that shift demand
    ★ Competitor price included in state space
    ★ Richer 5-dimensional state space
 
    State  : [eco_seats, biz_seats, days_left,
               competitor_price_idx, market_event_id]
    Action : MultiDiscrete([n_eco_prices, n_biz_prices])
    Reward : (eco_price × eco_bookings) + (biz_price × biz_bookings)
    """
 
    metadata = {'render_modes': ['human']}
 
    # Economy prices: $50 to $400 (8 levels)
    ECO_PRICES = [50, 100, 150, 200, 250, 300, 350, 400]
 
    # Business prices: $300 to $1200 (8 levels)
    BIZ_PRICES = [300, 450, 600, 750, 900, 1050, 1100, 1200]
 
    # Competitor prices (economy equivalent)
    COMPETITOR_PRICES = [80, 120, 160, 200, 240, 280, 320, 360]
 
    def __init__(self,
                 eco_seats   = 40,
                 biz_seats   = 10,
                 total_days  = 30,
                 render_mode = None):
 
        super().__init__()
        self.eco_seats_init  = eco_seats
        self.biz_seats_init  = biz_seats
        self.total_days      = total_days
        self.render_mode     = render_mode
 
        # ── Action Space ──────────────────────────────────
        # Agent picks: [economy_price_idx, business_price_idx]
        # Both independently chosen from their price lists
        self.action_space = spaces.MultiDiscrete(
            [len(self.ECO_PRICES), len(self.BIZ_PRICES)]
        )
 
        # ── State Space ───────────────────────────────────
        # [eco_seats, biz_seats, days_left,
        #  competitor_price_idx, market_event_id]
        self.observation_space = spaces.Box(
            low  = np.array([0, 0, 0, 0, 0],              dtype=np.float32),
            high = np.array([eco_seats, biz_seats,
                             total_days,
                             len(self.COMPETITOR_PRICES)-1,
                             len(MARKET_EVENTS)-1],        dtype=np.float32),
            dtype= np.float32
        )
 
        self._init_state()
 
    def _init_state(self):
        self.eco_seats        = self.eco_seats_init
        self.biz_seats        = self.biz_seats_init
        self.days_left        = self.total_days
        self.total_revenue    = 0.0
        self.eco_revenue      = 0.0
        self.biz_revenue      = 0.0
        self.market_event_id  = 0
        self.competitor_idx   = 3   # start at mid-range competitor price
        self.history          = []
 
    def _sample_market_event(self):
        """★ UNIQUE: Sample today's external market condition."""
        return int(np.random.choice(len(MARKET_EVENTS), p=EVENT_PROBS))
 
    def _update_competitor_price(self):
        """★ UNIQUE: Competitor price drifts randomly each day."""
        drift = np.random.choice([-1, 0, 0, 1])   # tends to stay stable
        self.competitor_idx = int(np.clip(
            self.competitor_idx + drift,
            0, len(self.COMPETITOR_PRICES) - 1
        ))
 
    def _segment_demand(self, price, days_left, seat_class, event_mult):
        """
        ★ UNIQUE: Compute bookings from each of 3 customer segments.
 
        Each segment has its own price sensitivity and booking window.
        The market event multiplier shifts all demand up or down.
        """
        max_price = max(self.BIZ_PRICES) if seat_class=='biz' else max(self.ECO_PRICES)
        total_bookings = 0
 
        for seg_name, seg in CUSTOMER_SEGMENTS.items():
 
            # Skip last-minute bookers if not near departure
            if seg['booking_window'] == 'late'  and days_left > 7:
                continue
            # Leisure bookers drop off sharply in last 3 days (already booked)
            if seg['booking_window'] == 'early' and days_left < 3:
                continue
 
            # Price sensitivity per segment
            price_factor = 1.0 - seg['price_sensitivity'] * (price / max_price)
            price_factor = max(0.0, price_factor)
 
            # Urgency (last-minute customers always appear near departure)
            if seg_name == 'last_minute':
                urgency = np.exp(-days_left / 3)
            elif seg_name == 'business':
                urgency = 0.6 + 0.4 * np.exp(-days_left / 15)
            else:
                urgency = 1.0 - 0.6 * np.exp(-days_left / 20)
 
            # Competitor effect — if we're cheaper, we get more demand
            our_price       = price
            competitor_eco  = self.COMPETITOR_PRICES[self.competitor_idx]
            comp_factor     = 1.0 + 0.3 * (competitor_eco - our_price) / max_price
            comp_factor     = float(np.clip(comp_factor, 0.5, 1.8))
 
            # Final probability for this segment
            prob = (seg['base_probability']
                    * price_factor
                    * urgency
                    * comp_factor
                    * event_mult
                    + np.random.uniform(-0.05, 0.05))
 
            prob = float(np.clip(prob, 0.0, 1.0))
 
            # Poisson arrivals per segment (max 3 per segment per day)
            arrivals = np.random.randint(0, 4)
            bookings = sum(np.random.random() < prob for _ in range(arrivals))
            total_bookings += bookings
 
        return total_bookings
 
    def _get_obs(self):
        return np.array([
            self.eco_seats,
            self.biz_seats,
            self.days_left,
            self.competitor_idx,
            self.market_event_id
        ], dtype=np.float32)
 
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._init_state()
        self.market_event_id = self._sample_market_event()
        self._update_competitor_price()
        return self._get_obs(), {}
 
    def step(self, action):
        eco_action, biz_action = int(action[0]), int(action[1])
        eco_price = self.ECO_PRICES[eco_action]
        biz_price = self.BIZ_PRICES[biz_action]
 
        # Today's market event
        event      = MARKET_EVENTS[self.market_event_id]
        event_mult = event['demand_multiplier']
 
        # Demand from all segments for each class
        eco_bookings = self._segment_demand(
            eco_price, self.days_left, 'eco', event_mult)
        biz_bookings = self._segment_demand(
            biz_price, self.days_left, 'biz', event_mult)
 
        # Cap at available seats
        eco_bookings = min(eco_bookings, self.eco_seats)
        biz_bookings = min(biz_bookings, self.biz_seats)
 
        # Revenue
        eco_rev = eco_price * eco_bookings
        biz_rev = biz_price * biz_bookings
        daily_rev = eco_rev + biz_rev
 
        # Update state
        self.eco_seats     -= eco_bookings
        self.biz_seats     -= biz_bookings
        self.total_revenue += daily_rev
        self.eco_revenue   += eco_rev
        self.biz_revenue   += biz_rev
        self.days_left     -= 1
 
        # Sample next day's event and competitor move
        self.market_event_id = self._sample_market_event()
        self._update_competitor_price()
 
        # Record
        self.history.append({
            'day'            : self.total_days - self.days_left,
            'days_left'      : self.days_left + 1,
            'eco_price'      : eco_price,
            'biz_price'      : biz_price,
            'competitor_price': self.COMPETITOR_PRICES[self.competitor_idx],
            'market_event'   : event['name'],
            'eco_bookings'   : eco_bookings,
            'biz_bookings'   : biz_bookings,
            'eco_rev'        : eco_rev,
            'biz_rev'        : biz_rev,
            'daily_revenue'  : daily_rev,
            'eco_seats_left' : self.eco_seats,
            'biz_seats_left' : self.biz_seats,
            'total_revenue'  : self.total_revenue
        })
 
        terminated = (self.days_left == 0 or
                      (self.eco_seats == 0 and self.biz_seats == 0))
        return self._get_obs(), daily_rev, terminated, False, {}
 
    def render(self):
        if self.render_mode == 'human':
            ev = MARKET_EVENTS[self.market_event_id]['name']
            print(f"  Day {self.total_days-self.days_left:>2} | "
                  f"Eco: {self.eco_seats:>2}seats | "
                  f"Biz: {self.biz_seats:>2}seats | "
                  f"Event: {ev:<18} | "
                  f"Revenue: ${self.total_revenue:,.0f}")
 
    def get_history_df(self):
        return pd.DataFrame(self.history)
 
 
print("✅ ContextualAirlinePricingEnv class defined")
print()
print("  State dimensions  : 5")
print("  Action dimensions : 2 (eco price + biz price)")
print("  Economy seats     : 40")
print("  Business seats    : 10")
print("  Selling horizon   : 30 days")
print()
print("  ★ Unique features active:")
print("     ★ Market Events (4 types with probabilities)")
print("     ★ 3 Customer Segments")
print("     ★ Competitor price drift in state")
print("     ★ Dual class inventory")
